# Lezione 09 — Topic modeling: LSA e LDA

**Corso:** NLP Classico con Python · **Blocco 4:** Significato e ricerca
**Tempo stimato:** 105 minuti · **Capitolo del manuale:** `manual/09_topic_modeling_IT.md`

## Cosa vedrai in questa lezione

- L'idea di **tema** come distribuzione sulle parole e di **documento** come miscela di temi, con un esempio che si calcola a mano.
- **LSA**: decomposizione ai valori singolari troncata (`TruncatedSVD`) su TF-IDF, e come si leggono (e come no) le sue dimensioni.
- **LDA** con `gensim` (`LdaModel`), che si addestra sui **conteggi** e non su TF-IDF: parole principali e miscela di temi di ogni documento, con seme fisso.
- Come i **nomi propri**, le stopword e la lemmatizzazione cambiano i temi.
- **Scegliere il numero di temi** con `CoherenceModel` (c_v e u_mass) e criterio umano, e vedere l'**instabilità** con altri semi e altre dimensioni di documento.
- Verificare in modo onesto se LSA e LDA **recuperano** i domini noti delle recensioni sintetiche (purezza e ARI).

**Requisiti:** lezioni 3 e 4 (stopword, TF-IDF) e 8 (probabilità; la 10 proseguirà con gli embedding). **Librerie:** `scikit-learn`, `gensim`, `nltk` (stopword), `spaCy` (solo per un confronto con i lemmi), `numpy`, `pandas`, `matplotlib`.
**Dati:** `datos/gutenberg/quijote_cap01-14.txt`, `datos/gutenberg/promessi_sposi_cap01-06.txt`, `datos/resenas_es.csv` e `datos/resenas_it.csv` (in Colab, carica la cartella `datos/`). I testi letterari sono di pubblico dominio e di due secoli fa o più: **non rappresentano lo spagnolo né l'italiano di oggi**. Tutte le cifre sono un *esempio illustrativo*.

---
## Cella 0 — Setup

In [ ]:
import subprocess, sys

def asegurar(paquete, modulo):
    """Installa il pacchetto solo se manca (idempotente e silenzioso)."""
    try:
        __import__(modulo)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", paquete], check=True)

asegurar("numpy", "numpy")
asegurar("pandas", "pandas")
asegurar("matplotlib", "matplotlib")
asegurar("scikit-learn", "sklearn")
asegurar("gensim", "gensim")
asegurar("nltk", "nltk")
asegurar("spacy", "spacy")
print("Librerie pronte.")

In [ ]:
import os
import re
import math
import random
import time
import warnings
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import nltk
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.cluster import KMeans
from sklearn.preprocessing import Normalizer
from sklearn.metrics import adjusted_rand_score
from gensim.corpora import Dictionary
from gensim.models import LdaModel, CoherenceModel

warnings.filterwarnings("ignore")
random.seed(42)                          # seme fisso
np.random.seed(42)
plt.rcParams["figure.figsize"] = (9, 4)
plt.rcParams["axes.grid"] = True

def cargar_datos(nombre):
    """Cerca un file di dati in datos/, ../datos/ e /content/datos/."""
    for base in ("datos", "../datos", "/content/datos"):
        ruta = os.path.join(base, nombre)
        if os.path.exists(ruta):
            return ruta
    # Copia pubblica (Colab): se non c'è in locale, scarica il file dal repository dei notebook.
    try:
        import urllib.request
        destino = os.path.join("datos", nombre)
        os.makedirs(os.path.dirname(destino), exist_ok=True)
        urllib.request.urlretrieve(f"https://raw.githubusercontent.com/RaffaeleBini/arcadia-notebooks/main/14-nlp-classico/datos/{nombre}", destino)
        return destino
    except Exception:
        pass
    raise FileNotFoundError(
        "Non trovo " + nombre + ". Carica la cartella datos/ del corso (in Colab: pannello dei file) o lasciala accanto al notebook.")

def cargar_stopwords(idioma):
    """Lista di stopword di NLTK (la scarica se manca)."""
    try:
        return set(stopwords.words(idioma))
    except LookupError:
        nltk.download("stopwords", quiet=True)
        return set(stopwords.words(idioma))

def tabla(cabeceras, filas, ancho=14):
    """Stampa una tabella di testo allineata."""
    print("".join(str(c).ljust(ancho) for c in cabeceras))
    print("-" * ancho * len(cabeceras))
    for fila in filas:
        print("".join((f"{x:.3f}" if isinstance(x, float) else str(x)).ljust(ancho) for x in fila))

print("Configurazione pronta.")

In [ ]:
ENCAB = re.compile(r"^(Capítulo|CAPITOLO)\b")   # dato-es
CORPORA = {"spa": ("gutenberg/quijote_cap01-14.txt", "spanish", "es_core_news_sm"),
           "ita": ("gutenberg/promessi_sposi_cap01-06.txt", "italian", "it_core_news_sm")}

def leer_texto(ruta):
    """Legge un estratto: toglie i segni _corsivo_ e le intestazioni di capitolo e restituisce un solo testo."""
    parrafos = [p.strip() for p in open(ruta, encoding="utf-8").read().replace("_", "").split("\n\n") if p.strip()]
    return " ".join(p for p in parrafos if not ENCAB.match(p))

def nombres_propios(texto, minimo=3, umbral=0.8):
    """Euristica: parole che hanno la maiuscola almeno l'80 % delle volte in cui compaiono fuori da inizio frase."""
    con_mayus, total = Counter(), Counter()
    for m in re.finditer(r"\w+", texto):
        if re.search(r"(^|[.!?…:]\s*[«—\"“]?\s*)$", texto[max(0, m.start() - 4):m.start()]):
            continue                                   # inizio di frase: la maiuscola non informa
        w = m.group()
        total[w.lower()] += 1
        con_mayus[w.lower()] += w[0].isupper()
    return {w for w in total if total[w] >= minimo and con_mayus[w] / total[w] >= umbral}

def trozos(tokens, n=300):
    """Divide una lista di token in pseudodocumenti consecutivi di n token (l'ultimo pezzo corto si scarta se ha meno di n/2)."""
    return [tokens[i:i + n] for i in range(0, len(tokens) - n // 2, n)]

def preparar(tokens, stop, nombres=frozenset(), lemas=None, minimo=3):
    """Minuscole, via stopword e parole corte, via nomi propri; se si danno i lemmi, si usano al posto delle forme."""
    fuente = lemas if lemas is not None else [w.lower() for w in tokens]
    return [w for w in fuente if w not in stop and len(w) >= minimo and w not in nombres]

def bolsa(docs, no_below=3, no_above=0.25):
    """Dizionario di gensim (filtrato per frequenza di documento) e conteggi per documento."""
    dic = Dictionary(docs)
    dic.filter_extremes(no_below=no_below, no_above=no_above)
    return dic, [dic.doc2bow(d) for d in docs]

def entrenar_lda(docs, k, semilla=0, no_below=3, no_above=0.25):
    """LDA di gensim sui CONTEGGI, con seme fisso."""
    dic, bow = bolsa(docs, no_below, no_above)
    m = LdaModel(bow, num_topics=k, id2word=dic, random_state=semilla, passes=10, iterations=50, eval_every=None)
    return m, dic, bow

def temas_lda(m, n=8):
    """Le n parole più probabili di ogni tema."""
    return [[w for w, _ in m.show_topic(i, n)] for i in range(m.num_topics)]

def mezcla(m, bow):
    """Matrice documento-tema (ogni riga somma 1)."""
    return np.array([[p for _, p in m.get_document_topics(b, minimum_probability=0.0)] for b in bow])

def jaccard_medio(A, B):
    """Per ogni tema di A, il miglior Jaccard con un tema di B (sulle sue parole principali); si fa la media."""
    A, B = [set(a) for a in A], [set(b) for b in B]
    return float(np.mean([max(len(a & b) / len(a | b) for b in B) for a in A]))

def coherencia(m, docs, dic, bow, medida="c_v"):
    """Coerenza di gensim (c_v usa una finestra scorrevole sui testi; u_mass usa i conteggi)."""
    if medida == "u_mass":
        return CoherenceModel(model=m, corpus=bow, dictionary=dic, coherence="u_mass").get_coherence()
    return CoherenceModel(model=m, texts=docs, dictionary=dic, coherence=medida, processes=1).get_coherence()

def pureza(y, grupos):
    """Purezza: frazione di documenti che appartengono alla classe maggioritaria del proprio gruppo."""
    total = 0
    for g in sorted(set(grupos)):
        total += max(Counter(yi for yi, gi in zip(y, grupos) if gi == g).values())
    return total / len(y)

---
## 1. Tema = distribuzione sulle parole; documento = miscela di temi

Un **tema** è una distribuzione di probabilità sul vocabolario: `P(parola | tema)`. Un **documento** è una **miscela** di temi, con proporzioni `θ` che sommano 1. La probabilità che una parola del documento sia `w` è

$$P(w\mid d)=\sum_{t} \theta_{d,t}\,P(w\mid t)$$

Piccolo esempio (inventato, *esempio illustrativo*): due temi su quattro parole. Un documento con `θ = (0,7 ; 0,3)`. Calcola a mano `P(gol | d)` prima di eseguire (0,7·0,5 + 0,3·0,1).

In [ ]:
# datos-es
# Due temi inventati su un vocabolario di 4 parole (una riga per tema; ogni riga somma 1)
VOCAB = ["gol", "liga", "sal", "receta"]      # in italiano: gol, campionato, sale, ricetta
PHI = np.array([[0.5, 0.3, 0.1, 0.1],          # tema 1 (P(parola | tema 1))
                [0.1, 0.1, 0.4, 0.4]])         # tema 2

In [ ]:
theta = np.array([0.7, 0.3])
p_doc = theta @ PHI                        # miscela: P(w | d) = Σ_t θ_t · P(w | t)
for w, p in zip(VOCAB, p_doc):
    print(f"P({w} | d) = {p:.2f}")
print("somma:", p_doc.sum())
assert np.isclose(p_doc[0], 0.7 * 0.5 + 0.3 * 0.1) and np.isclose(p_doc.sum(), 1.0)   # coincide con il calcolo a mano: 0,38

### Cosa osservi?

`P(gol | d) = 0,38`: compare molto nel tema 1 (0,5), poco nel 2 (0,1) e il documento è soprattutto del tema 1. La miscela è sempre una distribuzione (somma 1). **LDA fa il cammino inverso**: dato il testo, cerca `PHI` e `θ` che lo spieghino. Nulla nel modello dice cosa «significhi» ciascun tema: ci sono solo numeri.

---
## 2. Dati: pseudodocumenti letterari e nomi propri

Un capitolo intero è **un** documento: troppo pochi per un modello di temi. Tagliamo ogni opera in **pezzi consecutivi di 300 token** (pseudodocumenti). Decisioni che dichiariamo perché cambiano i temi:

- **Pezzi di lunghezza fissa**, senza rispettare paragrafi né capitoli (un tema può restare spezzato). L'ultimo pezzo si scarta se è troppo corto.
- **Nomi propri** (`Quijote`, `Sancho`, `Renzo`, `Lucia`…): dominano qualsiasi tema. Li rileviamo con un'**euristica** (parole che quasi sempre hanno la maiuscola fuori da inizio frase); ha falsi positivi (`ah`, `oh`, `Dio`) e non è perfetta.
- **Stopword** di NLTK, parole di meno di 3 lettere e **filtro di frequenza di documento** (via le parole che compaiono in meno di 3 pezzi o in più del 25 %).

In [ ]:
DATOS = {}
for idioma, (ruta, nombre_nltk, modelo_spacy) in CORPORA.items():
    texto = leer_texto(cargar_datos(ruta))
    tokens = re.findall(r"\w+", texto)
    DATOS[idioma] = dict(tokens=tokens, docs=trozos(tokens, 300), nombres=nombres_propios(texto), stop=cargar_stopwords(nombre_nltk))
    d = DATOS[idioma]
    mas_frecuentes = [w for w, _ in Counter(t.lower() for t in tokens if t.lower() in d["nombres"]).most_common(6)]
    print(idioma, "| tokens:", len(tokens), "| pseudodocumenti:", len(d["docs"]),
          "| nomi rilevati:", len(d["nombres"]), "|", mas_frecuentes)
    assert len(d["docs"]) > 90 and all(len(x) == 300 for x in d["docs"])

### Cosa osservi?

Escono poco più di 100 pseudodocumenti per opera: **decine**, non migliaia. È poco per un modello probabilistico (lo verificheremo con la stabilità). I nomi più frequenti sono quelli dei protagonisti, proprio ciò che finirà per «sequestrare» i temi.

In [ ]:
# Preparazione di riferimento: minuscole, senza stopword né parole corte, senza nomi propri
for idioma, d in DATOS.items():
    d["base"] = [preparar(x, d["stop"], d["nombres"]) for x in d["docs"]]
    print(idioma, "token per documento dopo la preparazione (media):", round(np.mean([len(x) for x in d["base"]]), 1))
    print("  ", d["base"][0][:12])

---
## 3. LSA: TF-IDF + SVD troncata

**LSA** (*Latent Semantic Analysis*) applica una **decomposizione ai valori singolari troncata** alla matrice documento-termine TF-IDF: `X ≈ U Σ Vᵀ` con solo `k` dimensioni. Ogni riga di `Vᵀ` è una **dimensione latente**: un vettore di pesi sulle parole, che possono essere **positivi o negativi**. Non è una distribuzione di probabilità: **non è un «tema» nel senso di LDA**.

Tre cose da sapere quando le leggi:

1. **Il segno è arbitrario**: moltiplicare una dimensione per −1 dà la stessa decomposizione. Si interpreta il contrasto tra parole di segno opposto, non il segno.
2. La prima dimensione di solito raccoglie **il livello generale delle parole** (pesi quasi tutti dello stesso segno) e non un tema.
3. Le altre sono **combinazioni ortogonali** che separano i documenti: possono mescolare argomenti diversi.

In [ ]:
def lsa(docs, k=5, semilla=0, **args):
    """TF-IDF su documenti già tokenizzati e SVD troncata; restituisce (svd, vocabolario, matrice dei documenti)."""
    tf = TfidfVectorizer(analyzer=lambda d: d, min_df=3, max_df=0.25)
    X = tf.fit_transform(docs)
    svd = TruncatedSVD(n_components=k, random_state=semilla, **args).fit(X)
    return svd, np.array(tf.get_feature_names_out()), X

def extremos(svd, vocab, i, n=6):
    """Parole con maggior peso positivo e negativo della dimensione i."""
    comp = svd.components_[i]
    orden = np.argsort(comp)
    return list(vocab[orden[::-1][:n]]), list(vocab[orden[:n]])

for idioma, d in DATOS.items():
    svd, vocab, X = lsa(d["base"])
    d["lsa"] = (svd, vocab, X)
    print("=====", idioma, "| matrice TF-IDF:", X.shape, "| valori singolari:", np.round(svd.singular_values_, 2))
    print("  frazione di pesi positivi per dimensione:", np.round((svd.components_ > 0).mean(axis=1), 2))
    for i in range(4):
        pos, neg = extremos(svd, vocab, i)
        print(f"  dimensione {i} (+)", pos)
        print("      (-)", neg)
    assert np.allclose(svd.components_ @ svd.components_.T, np.eye(5), atol=1e-6)   # le dimensioni sono ortonormali

### Cosa osservi?

- La dimensione 0 ha il valore singolare più grande e, nelle altre, i pesi si ripartiscono tra positivi e negativi (guarda la frazione di pesi positivi): la prima raccoglie soprattutto il «livello generale» delle parole, non un argomento concreto.
- Nelle altre dimensioni compaiono dei **poli**: un gruppo di parole positive contro un altro di negative. Leggile come un **contrasto** (per esempio, parole di un episodio contro un altro), non come un tema con etichetta.
- **Le etichette le mette la persona**: il modello consegna solo numeri. Qualsiasi nome tu dia loro è una tua interpretazione, discutibile.

### Il segno è arbitrario (e la soluzione non è unica con algoritmi casuali)

Adattiamo la stessa SVD con due semi e con l'algoritmo esatto (`arpack`) e confrontiamo le dimensioni con la loro **correlazione**: un valore vicino a −1 significa «la stessa dimensione con il segno invertito».

In [ ]:
for idioma, d in DATOS.items():
    svd0, vocab, X = d["lsa"]
    svd1 = TruncatedSVD(n_components=5, random_state=1).fit(X)
    svd2 = TruncatedSVD(n_components=5, algorithm="arpack").fit(X)
    corr = lambda a, b: [round(float(np.corrcoef(a.components_[i], b.components_[i])[0, 1]), 3) for i in range(5)]
    print(idioma, "| seme 0 contro seme 1:", corr(svd0, svd1), "| seme 0 contro arpack:", corr(svd0, svd2))
    assert abs(corr(svd0, svd1)[0]) > 0.9                     # la dimensione 0 è stabile salvo il segno

### Cosa osservi?

Le prime dimensioni coincidono (correlazione vicina a ±1), le ultime meno, e qualcuna può comparire con **segno negativo**. Conclusione pratica: fissa `random_state`, non attribuire significato al segno e diffida delle dimensioni di ordine alto (raccolgono sempre meno varianza).

Ora i pseudodocumenti nelle dimensioni 1 e 2, colorati per la loro **posizione nel libro** (da 0, inizio, a 1, fine).

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(10, 4))
for eje, (idioma, d) in zip(ejes, DATOS.items()):
    svd, vocab, X = d["lsa"]
    Z = svd.transform(X)
    pos = np.linspace(0, 1, len(Z))
    graf = eje.scatter(Z[:, 1], Z[:, 2], c=pos, cmap="viridis", s=25)
    eje.set_title(f"LSA — {idioma}")
    eje.set_xlabel("dimensione 1")
    eje.set_ylabel("dimensione 2")
fig.colorbar(graf, ax=ejes, label="posizione nel libro")
plt.show()

---
## 4. LDA con gensim: conteggi, seme e miscela di temi

**LDA** (*Latent Dirichlet Allocation*, Blei, Ng e Jordan, 2003) è un modello **generativo**: ogni tema è una distribuzione sulle parole e ogni documento una miscela di temi; l'addestramento cerca i parametri che meglio spiegano i conteggi. Per questo si addestra sui **conteggi di parole** (sacco di parole, `doc2bow`) e **non su TF-IDF** (i pesi TF-IDF non sono conteggi e il modello probabilistico perde senso).

Con `gensim`: `Dictionary` (vocabolario e filtro) → `doc2bow` → `LdaModel(num_topics=k, random_state=…)`. Fissiamo `random_state`, 10 passate (`passes`) e 50 iterazioni interne. Il **numero di temi `k` lo scegli tu**.

In [ ]:
K = 5
for idioma, d in DATOS.items():
    d["lda"] = entrenar_lda(d["base"], K, semilla=0)
    m, dic, bow = d["lda"]
    print("=====", idioma, "| vocabolario:", len(dic), "| documenti:", len(bow))
    for i, palabras in enumerate(temas_lda(m, 8)):
        print(f"  tema {i}:", " ".join(palabras))
    theta = mezcla(m, bow)
    assert theta.shape == (len(bow), K) and np.allclose(theta.sum(axis=1), 1.0, atol=1e-5)     # ogni riga è una distribuzione
    d["theta"] = theta
    print("  peso medio del tema dominante:", round(float(theta.max(axis=1).mean()), 3), "| documenti con tema dominante > 0,9:", int((theta.max(axis=1) > 0.9).sum()), "/", len(theta))
    for j in (0, 40, 80):
        print(f"  documento {j}:", np.round(theta[j], 2), "→ tema dominante", int(theta[j].argmax()))

### Cosa osservi?

- Ogni tema è una lista di parole con probabilità; **non ha un nome**. Leggilo e decidi tu se è coerente (per esempio, «una locanda, un oste, delle armi» potrebbe chiamarsi «scene della locanda»; un'altra persona potrebbe chiamarlo in un altro modo). Altri temi sono miscele senza un filo chiaro: con pseudodocumenti tagliati alla cieca è normale.
- Ogni documento ha una **miscela** (`θ`): la riga somma 1 e il «tema dominante» è solo il suo massimo. Qui quasi tutte le miscele sono molto concentrate su un tema (con ~140 parole per documento e i parametri predefiniti di `gensim`); con altri documenti o con `alpha` diverso sarebbero più ripartite.
- Ciò che vedi dipende da `k`, dal seme, dal filtro e dal pre-trattamento: lo verifichiamo ora.

---
## 5. Esperimento guidato: l'effetto dei nomi propri

Addestriamo LDA con gli stessi parametri ma **senza togliere i nomi propri** e contiamo quanti ce ne sono tra le 10 parole principali dei 5 temi (50 posizioni in totale).

In [ ]:
def nombres_en_top(m, nombres, n=10):
    return sum(w in nombres for palabras in temas_lda(m, n) for w in palabras)

for idioma, d in DATOS.items():
    d["con_nombres"] = [preparar(x, d["stop"]) for x in d["docs"]]
    m_con = entrenar_lda(d["con_nombres"], K, semilla=0)[0]
    m_sin = d["lda"][0]
    print(idioma, "| nomi nel top-10 con i nomi:", nombres_en_top(m_con, d["nombres"]),
          "| senza nomi:", nombres_en_top(m_sin, d["nombres"]))
    for i, palabras in enumerate(temas_lda(m_con, 8)):
        print(f"  tema {i}:", " ".join(palabras))
    assert nombres_en_top(m_sin, d["nombres"]) == 0

### Cosa osservi?

Con i nomi presenti, buona parte delle parole principali sono nomi di personaggi e luoghi: i «temi» diventano **chi compare** (scena del curato, scena di un personaggio) più che **di cosa si parla**. Toglierli non è «la cosa giusta»: è una **decisione** che dipende dalla domanda (se vuoi separare episodi per personaggi, conviene lasciarli). L'importante è dichiararla.

---
## 6. Esperimento guidato: il pre-trattamento cambia i temi

Confrontiamo quattro varianti (LDA, `k = 5`, seme 0) con quella di riferimento. Poiché il modello stesso cambia con il seme, includiamo un **riferimento di rumore**: la stessa preparazione con il seme 1.

- `base`: quella di prima (minuscole, senza stopword né nomi, filtro di frequenza).
- `sin_stop`: non si tolgono stopword né parole corte, e **nemmeno** si filtrano le parole molto frequenti.
- `con_nombres`: i nomi propri si lasciano.
- `lemas`: si usano i lemmi di spaCy (`es_core_news_sm` / `it_core_news_sm`) al posto delle forme.

`Jaccard` misura quanto i temi della variante assomigliano a quelli del riferimento (1 = uguali). La **coerenza c_v di varianti con vocabolari diversi non è confrontabile**: la includiamo per mostrare perché (e per vedere quanto varia con il seme).

In [ ]:
import subprocess, sys, spacy

def cargar_modelo(nombre):
    """Carica un modello spaCy e lo scarica se manca (idempotente)."""
    try:
        return spacy.load(nombre, disable=["parser", "ner"])
    except OSError:
        subprocess.run([sys.executable, "-m", "spacy", "download", "-q", nombre], check=True)
        return spacy.load(nombre, disable=["parser", "ner"])

t0 = time.time()
for idioma, (ruta, nombre_nltk, modelo_spacy) in CORPORA.items():
    d = DATOS[idioma]
    nlp = cargar_modelo(modelo_spacy)
    lemas = [[t.lemma_.lower() for t in doc if t.is_alpha] for doc in nlp.pipe([" ".join(x) for x in d["docs"]])]
    d["variantes"] = {
        "base": d["base"],
        "sin_stop": [preparar(x, set(), d["nombres"], minimo=1) for x in d["docs"]],
        "con_nombres": d["con_nombres"],
        "lemas": [preparar(x, d["stop"], d["nombres"], lemas=l) for x, l in zip(d["docs"], lemas)],
    }
print("lemmatizzazione fatta")

In [ ]:
def ajustar_variante(nombre, docs, semilla=0):
    no_above = 1.0 if nombre == "sin_stop" else 0.25        # in sin_stop non si filtrano le frequenti
    return entrenar_lda(docs, K, semilla=semilla, no_above=no_above)

for idioma, d in DATOS.items():
    print("=====", idioma)
    referencia = temas_lda(d["lda"][0], 10)
    filas = []
    modelos = {}
    for nombre, docs in list(d["variantes"].items()) + [("base_semilla1", d["base"])]:
        semilla = 1 if nombre == "base_semilla1" else 0
        m, dic, bow = ajustar_variante(nombre.replace("_semilla1", ""), docs, semilla)
        modelos[nombre] = m
        top = temas_lda(m, 10)
        filas.append([nombre, len(dic), round(jaccard_medio(referencia, top), 2),
                      sum(w in d["nombres"] for t in top for w in t), sum(w in d["stop"] for t in top for w in t),
                      round(coherencia(m, docs, dic, bow, "c_v"), 3)])
    tabla(["variante", "|V|", "Jaccard", "nomi", "stopwords", "c_v"], filas, ancho=15)
    for nombre in ("sin_stop", "con_nombres", "lemas"):
        print(f"  {nombre:12s} tema 0:", " ".join(temas_lda(modelos[nombre], 8)[0]))
    d["jac_ruido"] = filas[-1][2]

### Cosa osservi?

- `sin_stop`: i temi si riempiono di `de`, `che`, `la`… (50 posizioni su 50 sono stopword): senza filtro, LDA modella la grammatica, non gli argomenti.
- `con_nombres`: i temi includono nomi. Il suo c_v non è chiaramente più basso di quello di `base`, benché i temi siano meno informativi sull'argomento: **una coerenza alta non garantisce temi utili**.
- `lemas`: i temi cambiano (Jaccard basso), ma **anche il rumore del seme è grande** (riga `base_semilla1`): con così poco testo non si può affermare che una variante sia «migliore», solo che **il pre-trattamento e il seme spostano i temi**.
- Guarda il c_v di `base` contro `base_semilla1`: **la stessa preparazione, con un altro seme, cambia il c_v quanto una qualsiasi variante**. **Non confrontare** i c_v delle righe né tra lingue: ogni riga ha un vocabolario e un corpus diversi.

---
## 7. Scegliere il numero di temi: coerenza e criterio umano

La **coerenza** valuta un tema in base alla frequenza con cui co-occorrono le sue parole principali. Usiamo due misure di `gensim`, e bisogna **dichiarare quale**:

- **c_v**: finestra scorrevole sui testi e similarità del coseno tra vettori di informazione mutua normalizzata. Valori tipici tra 0 e 1.
- **u_mass**: si calcola con i conteggi del corpus; è negativa (più vicina a 0 = più coerente).

Nessuna è «la verità»: variano con il seme e con il filtro. Addestriamo `k = 2…10` con **3 semi** e mostriamo la media e l'intervallo. La coerenza **non si può confrontare tra corpus o lingue come se fosse assoluta**: ogni grafico ha la sua scala.

In [ ]:
KS = list(range(2, 11))
SEMILLAS = (0, 1, 2)
COH = {}
t0 = time.time()
for idioma, d in DATOS.items():
    cv = np.zeros((len(KS), len(SEMILLAS)))
    um = np.zeros((len(KS), len(SEMILLAS)))
    for i, k in enumerate(KS):
        for j, s in enumerate(SEMILLAS):
            m, dic, bow = entrenar_lda(d["base"], k, semilla=s)
            cv[i, j] = coherencia(m, d["base"], dic, bow, "c_v")
            um[i, j] = coherencia(m, d["base"], dic, bow, "u_mass")
    COH[idioma] = (cv, um)
    print("=====", idioma)
    tabla(["k", "c_v media", "c_v min", "c_v max", "u_mass media"],
          [[k, float(cv[i].mean()), float(cv[i].min()), float(cv[i].max()), float(um[i].mean())] for i, k in enumerate(KS)], ancho=16)
    print("k con c_v medio maggiore:", KS[int(cv.mean(axis=1).argmax())], "| k con u_mass medio maggiore:", KS[int(um.mean(axis=1).argmax())])
print("tempo:", round(time.time() - t0), "s")

fig, ejes = plt.subplots(1, 2, figsize=(10, 4))
for eje, (idioma, (cv, um)) in zip(ejes, COH.items()):
    eje.errorbar(KS, cv.mean(axis=1), yerr=[cv.mean(axis=1) - cv.min(axis=1), cv.max(axis=1) - cv.mean(axis=1)], marker="o", capsize=3)
    eje.set_title(f"c_v — {idioma}  (scala propria)")
    eje.set_xlabel("k")
    eje.set_ylabel("c_v")
plt.show()

### Cosa osservi?

- La curva è **piatta e rumorosa**: l'intervallo tra semi per uno stesso `k` è dello stesso ordine delle differenze tra `k` diversi. **Non c'è un `k` «giusto»** che la coerenza riveli con chiarezza.
- c_v e u_mass **non devono per forza coincidere** sul miglior `k` (confronta le ultime due righe per lingua); u_mass peggiora quasi sempre all'aumentare di `k`, quindi tende a favorire pochi temi.
- Criterio pratico: usa la coerenza per **scartare valori assurdi** e decidi con criterio umano tra 2 o 3 candidati, leggendo i temi (ciascuno racconta qualcosa di diverso?, ci sono temi spazzatura?). E il fatto che un `k` abbia c_v più alto in spagnolo che in italiano **non dice nulla** su quale lingua «si modelli meglio»: i corpus sono diversi.

---
## 8. Esperimento guidato: stabilità con altri semi e altre dimensioni di documento

Addestriamo cinque volte con semi 0-4 (`k = 5`) e misuriamo la somiglianza (Jaccard medio, sulle 10 parole principali) tra ogni coppia di esecuzioni. Poi ripetiamo con **pseudodocumenti più corti o più lunghi** (100, 300 e 900 token) per vedere l'effetto della dimensione e del numero di documenti.

In [ ]:
def estabilidad(docs, k=5, semillas=(0, 1, 2, 3, 4)):
    """Jaccard medio tra tutte le coppie di esecuzioni con semi diversi."""
    tops = [temas_lda(entrenar_lda(docs, k, semilla=s, no_below=3)[0], 10) for s in semillas]
    pares = [jaccard_medio(tops[i], tops[j]) for i in range(len(tops)) for j in range(len(tops)) if i < j]
    return float(np.mean(pares)), float(np.min(pares)), float(np.max(pares))

for idioma, d in DATOS.items():
    print("=====", idioma)
    filas = []
    for n in (100, 300, 900):
        docs_n = [preparar(x, d["stop"], d["nombres"]) for x in trozos(d["tokens"], n)]
        media, minimo, maximo = estabilidad(docs_n)
        filas.append([n, len(docs_n), media, minimo, maximo])
    tabla(["token/doc", "documenti", "Jaccard medio", "min", "max"], filas, ancho=18)
    d["estab"] = filas

### Cosa osservi?

Nessuna configurazione dà temi davvero stabili: il Jaccard medio tra semi resta lontano da 1 in tutte. Con documenti molto corti ci sono molti documenti ma ognuno ha poche parole; con documenti lunghi ce ne sono pochissimi. Con questo corpus piccolo **non c'è una combinazione magica**. Conseguenze pratiche: (1) fissa il seme e **documentalo**; (2) non presentare un solo addestramento come «i temi del libro»; (3) se un tema compare con più semi, è più affidabile di uno che compare una volta sola.

---
## 9. LSA e LDA recuperano i domini noti delle recensioni?

Le recensioni sintetiche hanno una colonna `dominio` (`app`, `electronica`, `hotel`, `restaurante`, `servicio`, `tienda_online`: 6 valori). **`dominio` non è l'etichetta di sentimento** (`etiqueta`): è l'argomento della recensione. Se un modello di temi funziona, i suoi gruppi dovrebbero assomigliare ai domini. Per misurarlo senza barare:

1. Si adatta il modello **senza usare `dominio`** con `k = 6` (il numero reale di domini: in un caso reale non lo sapresti).
2. LSA non dà gruppi, quindi si applica **KMeans** sulle sue coordinate normalizzate; in LDA si prende il **tema dominante**.
3. Si misurano **purezza** (frazione di recensioni della classe maggioritaria del proprio gruppo) e **ARI** (*adjusted Rand index*: 1 = stessi gruppi, ≈ 0 = come a caso, indipendente da come si numerano i gruppi).
4. Si confrontano con un'**assegnazione a caso** e con `etiqueta` (che non dovrebbe essere recuperata).

Usiamo il testo pulito (`texto_limpio`). Sono 300 recensioni per lingua.

In [ ]:
ARCHIVOS = {"spa": "resenas_es.csv", "ita": "resenas_it.csv"}   # dato-es

def docs_resenas(idioma, agrupar=1):
    """Recensioni tokenizzate; con agrupar > 1 si uniscono recensioni CONSECUTIVE dello stesso dominio (usa la colonna dominio: controllo ottimistico)."""
    df = pd.read_csv(cargar_datos(ARCHIVOS[idioma]))
    stop = DATOS[idioma]["stop"]
    if agrupar == 1:
        textos, dominios, etiquetas = df.texto_limpio.tolist(), df.dominio.tolist(), df.etiqueta.tolist()
    else:
        textos, dominios, etiquetas = [], [], []
        for dom, g in df.sort_values(["dominio", "id"]).groupby("dominio"):
            t = g.texto_limpio.tolist()
            for i in range(0, len(t) - agrupar + 1, agrupar):
                textos.append(" ".join(t[i:i + agrupar]))
                dominios.append(dom)
                etiquetas.append("-")
    docs = [[w for w in re.findall(r"\w+", x.lower()) if w not in stop and len(w) >= 3] for x in textos]
    return docs, dominios, etiquetas

def evaluar(y, grupos):
    return round(adjusted_rand_score(y, grupos), 3), round(pureza(y, grupos), 3)

def lsa_lda_resenas(idioma, agrupar=1, k=6, semillas=(0, 1, 2)):
    docs, y, e = docs_resenas(idioma, agrupar)
    Z = TruncatedSVD(n_components=k, random_state=0).fit_transform(TfidfVectorizer(analyzer=lambda d: d, min_df=2, max_df=0.5).fit_transform(docs))
    g_lsa = KMeans(n_clusters=k, n_init=10, random_state=0).fit_predict(Normalizer().fit_transform(Z))
    res = {"n": len(docs), "LSA": evaluar(y, g_lsa), "LSA_etiqueta": evaluar(e, g_lsa)[0] if agrupar == 1 else None}
    lda = []
    for s in semillas:
        m, dic, bow = entrenar_lda(docs, k, semilla=s, no_below=2, no_above=0.5)
        lda.append(evaluar(y, mezcla(m, bow).argmax(axis=1)))
    res["LDA"] = lda
    res["azar"] = evaluar(y, np.random.RandomState(0).randint(0, k, len(docs)))
    return res

RES = {}
for idioma in ARCHIVOS:
    RES[idioma] = lsa_lda_resenas(idioma)
    r = RES[idioma]
    print(idioma, "recensioni:", r["n"], "| (ARI, purezza) | LSA+KMeans", r["LSA"], "| LDA (3 semi)", r["LDA"], "| caso", r["azar"], "| ARI di LSA contro etiqueta (sentimento):", r["LSA_etiqueta"])

### Cosa osservi?

Né LSA né LDA recuperano i domini: l'ARI è quasi 0 (come il caso) e la purezza supera appena quella del caso. **Non è un errore del codice**: le recensioni sono molto corte (≈ 9-10 parole utili), molte non nominano alcun termine proprio del loro dominio (`Cuidado, que luego pasa lo que pasa`) e **il vocabolario di opinione si mescola tra domini**. Nemmeno un classificatore supervisionato con TF-IDF supera un'accuratezza modesta nel prevedere `dominio` (lo verifichiamo subito). Un modello di temi può recuperare solo ciò che è nel vocabolario.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

for idioma in ARCHIVOS:
    docs, y, e = docs_resenas(idioma)
    X = TfidfVectorizer(analyzer=lambda d: d, min_df=2).fit_transform(docs)
    exactitud = cross_val_score(LogisticRegression(max_iter=1000), X, y, cv=5).mean()
    print(idioma, "accuratezza supervisionata prevedendo dominio (CV 5):", round(float(exactitud), 3), "| caso:", round(1 / len(set(y)), 3))

### Controllo positivo (ottimistico): documenti più lunghi e omogenei

Per vedere se il metodo funziona quando **il dato lo permette**, costruiamo pseudodocumenti unendo **10 recensioni consecutive dello stesso dominio** (27 documenti per lingua, circa 90-100 parole utili). **Attenzione: questo usa `dominio` per costruire i documenti**, quindi il risultato è un **tetto ottimistico**, non una valutazione onesta di un caso reale.

In [ ]:
for idioma in ARCHIVOS:
    r = lsa_lda_resenas(idioma, agrupar=10)
    print(idioma, "documenti:", r["n"], "| (ARI, purezza) | LSA+KMeans", r["LSA"], "| LDA (3 semi)", r["LDA"], "| caso", r["azar"])
    RES[idioma]["agrupado"] = r

### Cosa osservi?

Con documenti lunghi e omogenei, **LSA + KMeans recupera quasi tutti i domini** (ARI vicino a 0,8-0,9), mentre **LDA resta molto peggiore** e cambia con il seme: con soli 27 documenti, LDA (un modello probabilistico con molti parametri) non ha dati sufficienti. Questo **non significa che LSA sia meglio di LDA in generale**: significa che, in un corpus piccolo di testi corti e con un `k` noto, la soluzione deterministica e più semplice può essere più robusta. E ricorda: il confronto degli ARI **tra spagnolo e italiano** dice solo come si comporta ciascun corpus sintetico (sono testi diversi), non quale lingua sia più facile.

---
## 10. Riproducibilità

Con `random_state` fisso, due addestramenti con gli stessi dati e parametri danno **esattamente** gli stessi temi. Lo verifichiamo, e verifichiamo anche che **un altro seme** li cambia.

In [ ]:
d = DATOS["spa"]
m1 = entrenar_lda(d["base"], K, semilla=0)[0]
m2 = entrenar_lda(d["base"], K, semilla=0)[0]
m3 = entrenar_lda(d["base"], K, semilla=1)[0]
print("stesso seme, temi identici:", np.allclose(m1.get_topics(), m2.get_topics()))
print("altro seme, temi identici:", np.allclose(m1.get_topics(), m3.get_topics()))
assert np.allclose(m1.get_topics(), m2.get_topics()) and not np.allclose(m1.get_topics(), m3.get_topics())

---
## Riepilogo

| Concetto | Idea chiave |
|---|---|
| Tema e documento | tema = distribuzione sulle parole; documento = miscela di temi (`P(w\|d) = Σ θ·P(w\|t)`) |
| LSA | SVD troncata su TF-IDF; dimensioni con segno arbitrario, non sono temi né hanno un'etichetta |
| LDA | modello probabilistico addestrato sui **conteggi**; `random_state` fisso; `k` lo scegli tu |
| Pre-trattamento | nomi propri, stopword, filtro di frequenza e lemmi **cambiano i temi**; si dichiara ogni decisione |
| Coerenza | c_v / u_mass: utile per scartare, rumorosa, **non confrontabile tra corpus né lingue** |
| Stabilità | altro seme, altra dimensione di documento, altri temi; corpus piccolo = temi poco affidabili |
| Recuperare temi noti | si misura con purezza e ARI; con testi corti non si recuperano; il controllo ottimistico usa la colonna `dominio` |
| Etichette | le mette la persona, non il modello |

**Prossima:** nella lezione 10 passiamo dai sacchi di parole ai **vettori densi** (*word embedding*: Word2Vec e fastText). Poi la pratica di questa lezione.